# Session 3: sliding, overturning and the middle third
**A. Real-world problem.** A small wall holds up a garden bed. It must not slide, must not
tip over about its toe, and its base must not lift off the ground at the heel.

**Plan.** (1) earth pressure with backfill slope, surcharge and water, (2) the two safety
factors and the base pressure, (3) validate against closed forms and moment equilibrium,
(4) sweep width, friction and slope, (5) see how water in the backfill destroys the margin.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

GAMMA_W = 9.81     # unit weight of water, kN/m^3

## Step 1: earth pressure coefficients
Rankine (smooth vertical wall, backfill rising at angle beta):
Ka = cos(b) * (cos(b) - sqrt(cos(b)^2 - cos(phi)^2)) / (cos(b) + sqrt(cos(b)^2 - cos(phi)^2)).
For beta = 0 it becomes (1 - sin phi) / (1 + sin phi). Coulomb allows a rough, leaning wall
(angle alpha from vertical, wall friction delta) and also reduces to Rankine when
alpha = beta = delta = 0.

In [ ]:
def ka_rankine(phi_deg, beta_deg=0.0):
    """Rankine active coefficient for a vertical wall and a backfill sloping up at beta."""
    phi, beta = np.radians(phi_deg), np.radians(beta_deg)
    if beta >= phi:
        raise ValueError("backfill slope must be smaller than phi")
    root = np.sqrt(np.cos(beta) ** 2 - np.cos(phi) ** 2)
    return float(np.cos(beta) * (np.cos(beta) - root) / (np.cos(beta) + root))


def ka_coulomb(phi_deg, delta_deg=0.0, alpha_deg=0.0, beta_deg=0.0):
    """Coulomb active coefficient. alpha: wall back leaning from vertical, delta: wall friction."""
    phi, delta, alpha, beta = (np.radians(v) for v in (phi_deg, delta_deg, alpha_deg, beta_deg))
    inner = np.sin(phi + delta) * np.sin(phi - beta) / (np.cos(alpha + delta) * np.cos(alpha - beta))
    return float(np.cos(phi - alpha) ** 2
                 / (np.cos(alpha) ** 2 * np.cos(alpha + delta) * (1.0 + np.sqrt(inner)) ** 2))

## Step 2: all the forces on the wall, per metre run
A rectangular gravity wall of width B and height H, concrete unit weight gc. Soil force
on the back is Rankine with backfill slope beta (force parallel to the slope, so it has a
horizontal part Ph = Pa cos(beta) and a vertical part Pv = Pa sin(beta), acting on the back
of the wall at the heel). Surcharge q (kPa) adds Ka q H. Water table at height hw above
the base adds hydrostatic push and an uplift triangle under the base (pressure gamma_w hw at
the heel, zero at the toe). Passive resistance in front of the toe is neglected.

In [ ]:
def horizontal_forces(H, phi, gamma, beta=0.0, q=0.0, hw=0.0, gsat=20.0):
    """List of (name, force kN/m, height above base m) of the horizontal push on the wall back."""
    ka = ka_rankine(phi, beta)
    d = H - hw                                   # depth of the dry zone
    gsub = gsat - GAMMA_W                        # submerged unit weight
    if beta != 0.0 and (hw > 0.0 or q > 0.0):
        raise ValueError("slope is not combined with water or surcharge in this model")
    if beta != 0.0:
        pa = 0.5 * ka * gamma * H ** 2
        return [("soil", pa * np.cos(np.radians(beta)), H / 3.0)]
    parts = [("soil dry zone, triangle", 0.5 * ka * gamma * d ** 2, hw + d / 3.0),
             ("soil dry zone, rectangle on wet zone", ka * gamma * d * hw, hw / 2.0),
             ("soil wet zone, triangle", 0.5 * ka * gsub * hw ** 2, hw / 3.0)]
    if hw > 0.0:
        parts.append(("water", 0.5 * GAMMA_W * hw ** 2, hw / 3.0))
    if q > 0.0:
        parts.append(("surcharge", ka * q * H, H / 2.0))
    return [p for p in parts if p[1] > 0.0]


def wall_state(B, H, mu, phi, gamma, gc=24.0, beta=0.0, q=0.0, hw=0.0, gsat=20.0):
    """Everything about the wall: forces, moments about the toe, factors of safety, base pressure."""
    parts = horizontal_forces(H, phi, gamma, beta, q, hw, gsat)
    ph = sum(f for _, f, _ in parts)
    mo = sum(f * z for _, f, z in parts)                       # overturning moment, kN m/m
    pv = 0.0
    if beta != 0.0:
        pa = ph / np.cos(np.radians(beta))
        pv = pa * np.sin(np.radians(beta))
    W = gc * B * H
    U = 0.5 * GAMMA_W * hw * B
    m_res = W * B / 2.0 + pv * B                               # resisting moment about the toe
    m_ot = mo + U * 2.0 * B / 3.0                              # overturning incl. uplift
    N = W + pv - U                                             # net vertical force on the base
    fs_slide = mu * N / ph if N > 0 else 0.0
    fs_ot = m_res / m_ot
    xbar = (m_res - m_ot) / N if N > 0 else -np.inf            # where N acts, from the toe
    e = B / 2.0 - xbar
    q_toe = N / B * (1.0 + 6.0 * e / B)
    q_heel = N / B * (1.0 - 6.0 * e / B)
    return {"Ph": ph, "Mo": mo, "W": W, "U": U, "Pv": pv, "N": N, "M_res": m_res, "M_ot": m_ot,
            "FS_slide": fs_slide, "FS_ot": fs_ot, "xbar": xbar, "e": e,
            "q_toe": q_toe, "q_heel": q_heel, "middle_third": bool(abs(e) <= B / 6.0 + 1e-12)}

## Step 3: pass or fail
Commonly used guidance (not a code): FS_slide at least about 1.5, FS_ot at least about 2.0,
and the resultant inside the middle third (e <= B/6), so the heel pressure is not negative.

In [ ]:
FS_SLIDE_MIN, FS_OT_MIN = 1.5, 2.0


def passes(s):
    return bool(s["FS_slide"] >= FS_SLIDE_MIN and s["FS_ot"] >= FS_OT_MIN and s["middle_third"])


def min_width(H, mu, phi, gamma, gc=24.0, hw=0.0, q=0.0, step=0.05, Bmax=10.0):
    """Smallest B (multiple of step) that passes all three checks, or None."""
    for k in range(1, int(round(Bmax / step)) + 1):
        B = round(k * step, 10)
        if passes(wall_state(B, H, mu, phi, gamma, gc, 0.0, q, hw)):
            return B
    return None

## Step 4: validation by numerical integration
Build the pressure profile point by point, integrate it for force and moment, and compare
with the closed forms. Then check that the base pressure carries the vertical load and
gives the right moment (equilibrium of the whole free body).

In [ ]:
def numeric_push(H, phi, gamma, q=0.0, hw=0.0, gsat=20.0, n=4000):
    """Integrate effective soil + water pressure along the wall. Returns (force, moment about base)."""
    ka = ka_rankine(phi)
    z = (np.arange(n) + 0.5) * H / n                           # depth of each slice centre
    dz = H / n
    d = H - hw
    sv = np.where(z <= d, gamma * z, gamma * d + (gsat - GAMMA_W) * (z - d))   # effective vertical stress
    p = ka * sv + q * ka + np.where(z > d, GAMMA_W * (z - d), 0.0)
    force = float(np.sum(p) * dz)
    moment = float(np.sum(p * (H - z)) * dz)
    return force, moment


def base_pressure_check(s, B, n=2000):
    """Integrate the linear base pressure: returns (force, moment about the toe)."""
    x = (np.arange(n) + 0.5) * B / n
    qx = s["q_toe"] + (s["q_heel"] - s["q_toe"]) * x / B
    return float(np.sum(qx) * B / n), float(np.sum(qx * x) * B / n)

## Step 5: see it
Left: the safety factors against wall width, with the two limits. Right: the same wall as the
water table rises in the backfill (the failure case).

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    H, mu, phi, gamma = 4.0, 0.5, 30.0, 18.0
    s = wall_state(2.0, H, mu, phi, gamma)
    print(f"Ka Rankine = {ka_rankine(30):.4f}, Coulomb(delta=0) = {ka_coulomb(30):.4f}, Coulomb(delta=20) = {ka_coulomb(30, 20):.4f}")
    print(f"design wall B=2 m: Pa={s['Ph']:.1f} kN/m  W={s['W']:.1f}  FS_slide={s['FS_slide']:.2f}  FS_ot={s['FS_ot']:.2f}  e={s['e']:.3f} m  B/6={2/6:.3f} m")
    f, m = numeric_push(H, phi, gamma)
    print(f"numeric push {f:.2f} kN/m at {m / f:.3f} m above base")
    w = wall_state(2.0, H, mu, phi, gamma, hw=4.0)
    print(f"saturated: Ph={w['Ph']:.2f}  uplift={w['U']:.2f}  FS_slide={w['FS_slide']:.2f}  FS_ot={w['FS_ot']:.3f}")
    print("min width dry:", min_width(H, mu, phi, gamma), "  min width, water 2 m:", min_width(H, mu, phi, gamma, hw=2.0), " water 4 m:", min_width(H, mu, phi, gamma, hw=4.0))

    Bs = np.linspace(0.8, 4.0, 65)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    fs = [wall_state(b, H, mu, phi, gamma) for b in Bs]
    ax[0].plot(Bs, [r["FS_slide"] for r in fs], label="sliding")
    ax[0].plot(Bs, [r["FS_ot"] for r in fs], label="overturning")
    ax[0].axhline(FS_SLIDE_MIN, color="C0", ls=":")
    ax[0].axhline(FS_OT_MIN, color="C1", ls=":")
    ax[0].set_xlabel("wall width B (m)")
    ax[0].set_ylabel("factor of safety (no unit)")
    ax[0].set_title("Dry backfill: H = 4 m, friction 0.5")
    ax[0].grid(alpha=0.3)
    ax[0].legend()
    hws = np.linspace(0, 4, 41)
    ws = [wall_state(2.0, H, mu, phi, gamma, hw=h) for h in hws]
    ax[1].plot(hws, [r["FS_slide"] for r in ws], label="sliding")
    ax[1].plot(hws, [r["FS_ot"] for r in ws], label="overturning")
    ax[1].axhline(1.0, color="red", ls="--", label="FS = 1")
    ax[1].set_xlabel("water level in backfill above base (m)")
    ax[1].set_ylabel("factor of safety (no unit)")
    ax[1].set_title("B = 2 m wall as the water rises")
    ax[1].grid(alpha=0.3)
    ax[1].legend()
    fig.tight_layout()
    fig.savefig(OUT / "safety_factors.png", dpi=120)

    mus = np.linspace(0.3, 0.7, 5)
    betas = np.linspace(0, 20, 5)
    tab = np.array([[wall_state(2.0, H, m_, phi, gamma, beta=b_)["FS_slide"] for b_ in betas] for m_ in mus])
    fig2, ax2 = plt.subplots(figsize=(5.5, 4))
    im = ax2.imshow(tab, origin="lower", aspect="auto", extent=[-2.5, 22.5, 0.25, 0.75], cmap="viridis")
    ax2.set_xlabel("backfill slope beta (degrees)")
    ax2.set_ylabel("base friction coefficient (no unit)")
    ax2.set_title("FS_slide, B = 2 m (colour)")
    fig2.colorbar(im, ax=ax2, label="FS_slide")
    fig2.tight_layout()
    fig2.savefig(OUT / "sweep_slide.png", dpi=120)
    print(np.round(tab, 2))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `wall_state` to print FS_slide and FS_ot for B = 2.5 m, H = 4 m with a 10 kPa surcharge, and say whether the middle-third rule holds.
2. **Stretch.** Sweep `phi` from 25 to 35 degrees with `min_width` and print the minimum width for each; compare with 2.25 m at 25 degrees and 1.85 m at 35 degrees.
3. **Explain.** Explain why the dry design wall of the worked example (B = 2 m) is only just acceptable even though both factors of safety look comfortable.

## Assessment
- **A. Conceptual.** Why does water in the backfill reduce both the sliding and the overturning factors, and why is drainage behind a wall so important?
- **B. Numerical.** A wall 3 m high and 1.6 m wide (concrete 24 kN/m^3) holds dry soil (gamma = 18, phi = 30 degrees), base friction 0.5. Find FS_slide, FS_ot, the eccentricity and whether the resultant is in the middle third.
- **C. Graph and simulation interpretation.** In the right panel of the first figure (a 2 m wide wall), at what water levels does the sliding factor fall below 1.5 and 1? At what level does the overturning factor fall below 2 and 1? Which factor fails first?
- **D. Design and optimisation challenge.** Design: for a 4 m wall with base friction 0.5 (phi = 30 degrees), find the smallest width in steps of 0.05 m that passes all three checks (FS_slide >= 1.5, FS_ot >= 2.0, e <= B/6) for a dry backfill, and with water 2 m deep. Discuss the cost of poor drainage.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def _cross(L, key, target, B=2.0):
    lo, hi = 0.0, 4.0
    for _ in range(60):
        m = (lo + hi) / 2
        if L.wall_state(B, 4.0, 0.5, 30.0, 18.0, hw=m)[key] > target:
            lo = m
        else:
            hi = m
    return lo


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # earth pressure coefficients
    checks.append(("Rankine Ka for beta = 0 equals (1 - sin phi)/(1 + sin phi) for several phi",
                   all(abs(L.ka_rankine(p) - (1 - np.sin(np.radians(p))) / (1 + np.sin(np.radians(p)))) < 1e-12 for p in (20, 30, 40))))
    checks.append(("Coulomb reduces to Rankine for delta = alpha = beta = 0",
                   all(abs(L.ka_coulomb(p) - L.ka_rankine(p)) < 1e-12 for p in (20, 30, 40))))
    checks.append(("Coulomb with delta = beta, vertical wall, equals Rankine with that slope (10 degrees: 0.3495)",
                   abs(L.ka_coulomb(30, 10, 0, 10) - L.ka_rankine(30, 10)) < 1e-12 and abs(L.ka_rankine(30, 10) - 0.3495) < 1e-4))
    checks.append(("wall friction delta = 20 degrees lowers Ka from 0.3333 to 0.2973 (about 11 percent)",
                   abs(L.ka_coulomb(30, 20) - 0.2973) < 1e-4 and abs(1 - L.ka_coulomb(30, 20) / L.ka_rankine(30) - 0.1080) < 1e-3))
    checks.append(("a backfill slope raises Rankine Ka: 0.3333, 0.3495, 0.4142 at 0, 10, 20 degrees",
                   abs(L.ka_rankine(30, 0) - 1 / 3) < 1e-4 and abs(L.ka_rankine(30, 10) - 0.3495) < 1e-4
                   and abs(L.ka_rankine(30, 20) - 0.4142) < 1e-4))
    try:
        L.ka_rankine(30, 30)
        bad = True
    except ValueError:
        bad = False
    checks.append(("edge: a slope equal to phi is refused (no active state)", not bad))

    # worked example: B = 2, H = 4, mu = 0.5
    s = L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0)
    checks.append(("worked example: Pa = 48 kN/m, W = 192 kN/m, FS_slide = 2.00, FS_ot = 3.00",
                   abs(s["Ph"] - 48) < 1e-9 and abs(s["W"] - 192) < 1e-9 and abs(s["FS_slide"] - 2.0) < 1e-9
                   and abs(s["FS_ot"] - 3.0) < 1e-9))
    checks.append(("worked example: resultant at 0.667 m from the toe, e = 0.333 m = B/6, heel pressure zero",
                   abs(s["xbar"] - 2 / 3) < 1e-9 and abs(s["e"] - 1 / 3) < 1e-9 and abs(s["q_heel"]) < 1e-9
                   and s["middle_third"] and abs(s["q_toe"] - 192.0) < 1e-9))
    # hand calculation, B = 1.6 m, H = 3 m
    h = L.wall_state(1.6, 3.0, 0.5, 30.0, 18.0)
    checks.append(("hand calculation: H = 3, B = 1.6: FS_slide 2.13, FS_ot 3.41, e = 0.234 m < B/6 = 0.267 m",
                   abs(h["FS_slide"] - 2.1333) < 1e-3 and abs(h["FS_ot"] - 3.4133) < 1e-3
                   and abs(h["e"] - 0.2344) < 1e-3 and h["middle_third"]))

    # validation: moment equilibrium of the full free body
    ok_eq = True
    for kw in ({}, {"q": 10.0}, {"hw": 2.0}, {"hw": 4.0}, {"beta": 10.0}):
        st = L.wall_state(2.5, 4.0, 0.5, 30.0, 18.0, **kw)
        F, M = L.base_pressure_check(st, 2.5)
        ok_eq &= abs(F - st["N"]) < 1e-6 * max(1, st["N"]) and abs(M - st["N"] * st["xbar"]) < 1e-6 * max(1, abs(M))
        ok_eq &= abs(M - (st["M_res"] - st["M_ot"])) < 1e-6 * max(1, abs(M))
    checks.append(("base pressure carries the vertical load and the moment about the toe (dry, surcharge, water, slope)", bool(ok_eq)))

    # validation: numerical pressure integration against the closed forms
    ok_int = True
    for kw in ({}, {"q": 10.0}, {"hw": 2.0}, {"hw": 4.0}, {"q": 10.0, "hw": 1.0}):
        parts = L.horizontal_forces(4.0, 30.0, 18.0, **kw)
        f_cf = sum(f for _, f, _ in parts)
        m_cf = sum(f * z for _, f, z in parts)
        f_n, m_n = L.numeric_push(4.0, 30.0, 18.0, **kw)
        ok_int &= abs(f_n - f_cf) < 1e-3 and abs(m_n - m_cf) < 1e-3
    checks.append(("numerical integration of the pressure matches the closed-form force and moment", bool(ok_int)))
    sat = L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0, hw=4.0)
    checks.append(("saturated backfill: horizontal force 105.65 kN/m, of which water 78.48", abs(sat["Ph"] - 105.65) < 0.01
                   and abs(0.5 * 9.81 * 16 - 78.48) < 1e-9))
    checks.append(("zero water level gives the dry result", abs(L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0, hw=0.0)["Ph"] - 48) < 1e-9))

    # limits and guidance
    checks.append(("the dry B = 2 m wall just passes (sliding 2.0, overturning 3.0, middle third)", L.passes(s)))
    checks.append(("narrow wall B = 1.5 m: sliding exactly 1.5 but overturning only 1.69 and e outside B/6",
                   abs(L.wall_state(1.5, 4.0, 0.5, 30.0, 18.0)["FS_slide"] - 1.5) < 1e-9
                   and abs(L.wall_state(1.5, 4.0, 0.5, 30.0, 18.0)["FS_ot"] - 1.6875) < 1e-9
                   and not L.passes(L.wall_state(1.5, 4.0, 0.5, 30.0, 18.0))))
    checks.append(("sliding and overturning rise with width and friction",
                   L.wall_state(3.0, 4.0, 0.5, 30.0, 18.0)["FS_ot"] > s["FS_ot"] and L.wall_state(2.0, 4.0, 0.6, 30.0, 18.0)["FS_slide"] > s["FS_slide"]))
    sweep = [L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0, beta=b) for b in (0, 10, 20)]
    checks.append(("backfill slope sweep: sliding 2.00, 2.03, 1.89 at 0, 10, 20 degrees (changes less than 6 percent)",
                   abs(sweep[0]["FS_slide"] - 2.0) < 1e-9 and abs(sweep[1]["FS_slide"] - 2.025) < 1e-3
                   and abs(sweep[2]["FS_slide"] - 1.895) < 1e-3 and max(abs(r["FS_slide"] - 2.0) for r in sweep) < 0.11))
    checks.append(("slope sweep: horizontal push 48.0 kN/m rising to 56.05 kN/m at 20 degrees (Ka 0.4142)",
                   abs(sweep[2]["Ph"] - 56.05) < 0.01))

    # sensitivity to phi
    p25 = L.wall_state(2.0, 4.0, 0.5, 25.0, 18.0)
    p35 = L.wall_state(2.0, 4.0, 0.5, 35.0, 18.0)
    checks.append(("phi 25 versus 35 degrees for the B = 2 m wall: sliding 1.64 versus 2.46, overturning 2.46 versus 3.69",
                   abs(p25["FS_slide"] - 1.6426) < 1e-3 and abs(p35["FS_slide"] - 2.4601) < 1e-3
                   and abs(p25["FS_ot"] - 2.4639) < 1e-3 and abs(p35["FS_ot"] - 3.6902) < 1e-3))
    # surcharge and water lead out of the middle third
    sur = L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0, q=10.0)
    checks.append(("a 10 kPa surcharge on the B = 2 m wall: e = 0.472 m exceeds B/6 and the heel pressure is negative (-40 kPa)",
                   abs(sur["e"] - 0.4722) < 1e-3 and not sur["middle_third"] and abs(sur["q_heel"] + 40.0) < 0.01))

    # failure case: water in the backfill
    fs = [L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0, hw=w) for w in (0.0, 1.0, 2.0, 3.0, 4.0)]
    checks.append(("water at 0, 1, 2, 3, 4 m: FS_ot 3.00, 2.45, 1.92, 1.41, 0.99",
                   all(abs(a - b) < 6e-3 for a, b in zip([r["FS_ot"] for r in fs], [3.0, 2.453, 1.924, 1.415, 0.994]))))
    checks.append(("failure case: with the backfill saturated, uplift is 39.24 kN/m, FS_slide 0.72 and FS_ot 0.994 (below 1)",
                   abs(sat["U"] - 39.24) < 1e-9 and abs(sat["FS_slide"] - 0.723) < 2e-3 and sat["FS_ot"] < 1.0
                   and abs(sat["FS_ot"] - 0.994) < 1e-3 and sat["xbar"] < 0))
    checks.append(("sliding falls below 1.5 at a water level of 1.70 m and below 1 at 3.03 m",
                   abs(_cross(L, "FS_slide", 1.5) - 1.7018) < 5e-3 and abs(_cross(L, "FS_slide", 1.0) - 3.0321) < 5e-3))
    checks.append(("overturning falls below 2 at 1.86 m and below 1 at 3.98 m",
                   abs(_cross(L, "FS_ot", 2.0) - 1.8577) < 5e-3 and abs(_cross(L, "FS_ot", 1.0) - 3.9831) < 5e-3))
    checks.append(("safety factors fall as the water rises",
                   all(fs[i]["FS_slide"] > fs[i + 1]["FS_slide"] and fs[i]["FS_ot"] > fs[i + 1]["FS_ot"] for i in range(4))))

    # design widths
    checks.append(("design: smallest passing width for H = 4 m is 2.00 dry, 2.45 with water at 2 m, 4.15 saturated",
                   L.min_width(4.0, 0.5, 30.0, 18.0) == 2.0 and L.min_width(4.0, 0.5, 30.0, 18.0, hw=2.0) == 2.45
                   and L.min_width(4.0, 0.5, 30.0, 18.0, hw=4.0) == 4.15))
    w25 = L.min_width(4.0, 0.5, 25.0, 18.0)
    w35 = L.min_width(4.0, 0.5, 35.0, 18.0)
    checks.append(("design: phi 25 degrees needs 2.25 m, phi 35 degrees needs 1.85 m", w25 == 2.25 and w35 == 1.85))
    checks.append(("challenge: B = 2.5 m, friction 0.5, phi 30, water at 2 m passes; B = 2.4 m does not",
                   L.passes(L.wall_state(2.5, 4.0, 0.5, 30.0, 18.0, hw=2.0)) and not L.passes(L.wall_state(2.4, 4.0, 0.5, 30.0, 18.0, hw=2.0))
                   and abs(L.wall_state(2.5, 4.0, 0.5, 30.0, 18.0, hw=2.0)["FS_slide"] - 1.726) < 2e-3))

    # extra numbers quoted in the text
    w3 = L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0, hw=3.0)
    checks.append(("water 3 m on the B = 2 m wall: FS_slide 1.01 and FS_ot 1.41", abs(w3["FS_slide"] - 1.011) < 2e-3 and abs(w3["FS_ot"] - 1.415) < 2e-3))
    checks.append(("hand calculation: H = 3, B = 1.6 base pressures 135.3 kPa (toe) and 8.7 kPa (heel)",
                   abs(h["q_toe"] - 135.28) < 0.01 and abs(h["q_heel"] - 8.72) < 0.01))
    checks.append(("sliding factor is proportional to friction: 1.2 at 0.3 and 2.8 at 0.7 (B = 2 m)",
                   abs(L.wall_state(2.0, 4.0, 0.3, 30.0, 18.0)["FS_slide"] - 1.2) < 1e-9 and abs(L.wall_state(2.0, 4.0, 0.7, 30.0, 18.0)["FS_slide"] - 2.8) < 1e-9))
    d195 = L.wall_state(1.95, 4.0, 0.5, 30.0, 18.0)
    checks.append(("dry design is controlled by the middle-third rule: at B = 1.95 m both factors pass but e > B/6",
                   d195["FS_slide"] >= 1.5 and d195["FS_ot"] >= 2.0 and not d195["middle_third"]))
    checks.append(("the undrained 2.45 m wall needs 22.5 percent more width than the dry 2.00 m wall", abs(2.45 / 2.0 - 1.225) < 1e-12))
    # mini-project: garden wall
    g = {"H": 1.2, "mu": 0.45, "phi": 30.0, "gamma": 18.0, "gc": 22.0}
    wd = L.min_width(g["H"], g["mu"], g["phi"], g["gamma"], g["gc"], q=5.0)
    w1 = L.min_width(g["H"], g["mu"], g["phi"], g["gamma"], g["gc"], hw=0.6, q=5.0)
    w2 = L.min_width(g["H"], g["mu"], g["phi"], g["gamma"], g["gc"], hw=1.2, q=5.0)
    checks.append(("mini-project: garden wall 1.2 m, 5 kPa surcharge needs B = 0.85 m dry, 1.10 m with water at 0.6 m, 1.90 m saturated",
                   wd == 0.85 and w1 == 1.1 and w2 == 1.9))
    a = L.wall_state(0.8, 1.2, 0.45, 30.0, 18.0, 22.0, q=5.0)
    b = L.wall_state(0.85, 1.2, 0.45, 30.0, 18.0, 22.0, q=5.0)
    checks.append(("mini-project: at 0.80 m both factors pass (1.50, 2.89) but the middle-third rule fails; at 0.85 m it passes (1.60, 3.26)",
                   a["FS_slide"] >= 1.5 and a["FS_ot"] >= 2.0 and not a["middle_third"]
                   and abs(a["FS_slide"] - 1.504) < 2e-3 and abs(a["FS_ot"] - 2.885) < 2e-3
                   and L.passes(b) and abs(b["FS_slide"] - 1.598) < 2e-3 and abs(b["FS_ot"] - 3.257) < 2e-3))
    # edge: a wall lifted off its base
    checks.append(("edge: water level equal to the wall height: overturning moment 140.87 kN m/m (= 105.65 * 4/3)",
                   abs(L.wall_state(2.0, 4.0, 0.5, 30.0, 18.0, hw=4.0)["Mo"] - 140.871) < 1e-3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
